# Rehearsal fine-tuning: results tables

Builds the three tables (zero-shot, target-only, replay) from the runs produced by
`06.finetune_models/finetune_rehearsal_new_method.ipynb`, plus forgetting/recovery CSVs.
Outputs go to `new_method/<output_dir>/tables/`.

Positive `forgetting` (zero-shot minus target-only) is a drop; positive `recovery`
(replay minus target-only) is an improvement.

In [ ]:
output_dir = None  # None keeps new_method/config.json's output_dir; must match stage 06

In [ ]:
import os
import sys
from pathlib import Path

# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

PIPELINE = Path.cwd()
ROOT = PIPELINE / "new_method"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lidlab.data import load_config

c = load_config(ROOT / "config.json")
if output_dir:
    c["output_dir"] = output_dir

from lidlab.metrics import make_tables

In [ ]:
tables = make_tables(c)
for phase, table in tables.items():
    print(phase)
    display(table.round(4))
print("Saved CSV and LaTeX tables to:", ROOT / c["output_dir"] / "tables")

In [ ]:
import pandas as pd

for column in ("Macro F1 (target 3)", "Macro F1 (replay 8)", "Macro F1 (all 11)"):
    summary = pd.DataFrame({phase: table[column] for phase, table in tables.items()})
    summary["forgetting"] = summary.zero_shot - summary.target_only
    summary["recovery"] = summary.replay - summary.target_only
    summary["residual_drop"] = summary.zero_shot - summary.replay
    print(column)
    display(summary.round(4))